In [6]:
words = ['Apple','banana','APPLE','Cherry','banana']
lower_words = [w.lower() for w in words]
print(lower_words)
word_counts = {w.lower():words.count(w) + words.count(w.upper())
               for w in set(words)}
print(word_counts)
from collections import Counter
word_counts = dict(Counter(w.lower() for w in words))
print(word_counts)
first_letters = {w[0].upper() for w in words}
print(first_letters)

['apple', 'banana', 'apple', 'cherry', 'banana']
{'apple': 2, 'cherry': 1, 'banana': 2}
{'apple': 2, 'banana': 2, 'cherry': 1}
{'A', 'C', 'B'}


In [1]:
matrix = [[1,2,3],[4,5,6],[7,8,9]]
flat_events = [num for row in matrix for num in row if num % 2 ==0]
print(flat_events)

[2, 4, 6, 8]


In [ ]:
list1 = [1,2,3]
list2 = ['a','b','c']
import itertools
pairs = list(zip(list1,list2))
print(pairs)
def fibonacci():
    a,b = 0,1
    while True:
        yield a
        a,b = b,a+b
first_10_fib = list(itertools.islice(fibonacci(),10))
print(first_10_fib)
numbers = [1,2,3,4,5,6,7,8]
sorted_nums = sorted(numbers,key=lambda x :x % 2)
groups= {k:list(v) for k,v in itertools.groupby(sorted_nums,key=lambda x :x % 2)}
print(sorted_nums)
print(groups)



[(1, 'a'), (2, 'b'), (3, 'c')]
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
[2, 4, 6, 8, 1, 3, 5, 7]
{0: [2, 4, 6, 8], 1: [1, 3, 5, 7]}


In [7]:
users = [
    {'name':'alice','email':'alice@example.com','active':True},
    {'name':'bob','email':'bob@test.com','active':False},
    {'name':'charlie','email':'charlie@demo.com','active':True},
]
emails = ','.join(
    user['email'].lower()
    for user in users 
    if user['active']
)
print(emails)

alice@example.com,charlie@demo.com


In [ ]:
from dataclasses import dataclass,field
from uuid import uuid4
@dataclass
class OrderItem:
    name:str
    price:float
@dataclass
class Order:
    items:list[OrderItem] = field(default_factory=list)
    id:str = field(default_factory=lambda:str(uuid4()))
    total:float = field(init=False,default=0.0)
    def __post_init__(self):
        self.total = sum(item.price for item in self.items)
        if self.total <= 0 and self.items:
            raise ValueError(f"订单总价必须 > 0,当前: {self.total}")
order = Order(items=[
    OrderItem('Python Book',59.9),
    OrderItem('Sticker',9.9)
])
print(order.id)
print(order.total)





00c0370a-9273-404b-9963-884f8ef038fc
0.0


In [10]:
from typing import Protocol,runtime_checkable
@runtime_checkable
class Drawable(Protocol):
    def draw(self) -> None:...
class Circle:
    def draw(self) -> None:
        print("⭕ Drawing circle")
class Button:
    def draw(self) ->None:
        print("🔲 Drawing button")
    def click(self) -> None:
        print('Clicked!')
def render(obj:Drawable) -> None:
    obj.draw()
render(Circle())
render(Button())
print(isinstance(Circle(),Drawable))


⭕ Drawing circle
🔲 Drawing button
True


In [13]:
from datetime import datetime
from collections import Counter
LOG_LINES = [
    "2026-07-06 08:01:23 ERROR Connection refused",
    "2026-07-06 08:02:45 INFO Request OK",
    "2026-07-06 09:15:00 ERROR Timeout",
    "2026-07-06 09:30:12 ERROR Disk full",
    "2026-07-06 10:00:00 INFO Startup complete",
]
def read_lines(lines):
    for line in lines:
        yield line.strip()
def filter_errors(lines):
    for line in lines:
        if 'ERROR' in line:
            yield line
def extract_timestamps(lines):
    for line in lines:
        ts_str = line[:19]
        yield datetime.strptime(ts_str,"%Y-%m-%d %H:%M:%S")
def count_by_hour(timestamps):
    return Counter(ts.strftime("%Y-%m-%d %H:00") for ts in timestamps)
pipeline = read_lines(LOG_LINES)
pipeline = filter_errors(pipeline)
pipeline = extract_timestamps(pipeline)
result = count_by_hour(pipeline)
print(result)

Counter({'2026-07-06 09:00': 2, '2026-07-06 08:00': 1})


In [22]:
from dataclasses import dataclass, field
from typing import Protocol, Callable, Any
from functools import partial

# 1. 定义验证器协议
class Validator(Protocol):
    def __call__(self, value: Any) -> str | None: ...  # 返回错误消息或 None

# 2. 内置验证器工厂（用 partial 固化参数）
def min_length(value: str, n: int) -> str | None:
    return f"长度不能少于{n}" if len(value) < n else None

def max_value(value: int, limit: int) -> str | None:
    return f"不能超过{limit}" if value > limit else None

min_3_chars: Validator = partial(min_length, n=3)
max_100: Validator = partial(max_value, limit=100)

# 3. 字段规则定义
@dataclass
class FieldRule:
    name: str
    validators: list[Validator] = field(default_factory=list)

# 4. 验证引擎
@dataclass
class ConfigSchema:
    fields: list[FieldRule] = field(default_factory=list)
    
    def validate(self, config: dict) -> dict[str, list[str]]:
        """验证配置，返回 {字段名: [错误列表]}"""
        errors = {}
        for rule in self.fields:
            value = config.get(rule.name)
            field_errors = [
                msg for v in rule.validators 
                if (msg := v(value)) is not None  # 🆕 海象运算符！
            ]
            if field_errors:
                errors[rule.name] = field_errors
        return errors

# 5. 声明式使用
schema = ConfigSchema(fields=[
    FieldRule("username", [min_3_chars]),
    FieldRule("age", [max_100]),
    FieldRule("email", [min_3_chars]),
])

bad_config = {"username": "ab", "age": 150, "email": ""}
errors = schema.validate(bad_config)
print(errors)
# {'username': ['长度不能少于3'], 'age': ['不能超过100'], 'email': ['长度不能少于3']}

{'username': ['长度不能少于3'], 'age': ['不能超过100'], 'email': ['长度不能少于3']}
